# 04 · TransNet 100 epoch 경향성 실험

저자 TransNet 저장소의 모델 구조를 사용합니다. 입력을 저자 구현이 요구하는 `[N, 2048]`로 바꾸는 어댑터를 포함합니다. 이 노트북은 test split을 읽지 않고 train/validation만 사용합니다. 기본값은 학습 꺼짐이며, 먼저 난수 입력으로 입출력 shape를 확인합니다. 이 100 epoch 결과는 경향성 확인용입니다.

In [ ]:
from pathlib import Path
import sys, shutil, torch
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT_OVERRIDE = None  # 필요하면 프로젝트 경로 지정
if PROJECT_ROOT_OVERRIDE: PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
DATA_DIR = PROJECT_ROOT / 'Data'
assert DATA_DIR.is_dir(), f'Data 폴더를 찾을 수 없음: {DATA_DIR}'
if IN_COLAB:
    ACTIVE_DATA_DIR = Path('/content/cost2100_indoor'); ACTIVE_DATA_DIR.mkdir(exist_ok=True)
    for name in ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']:
        src, dst = DATA_DIR / name, ACTIVE_DATA_DIR / name
        assert src.is_file(), f'필수 파일 없음: {src}'
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size: shutil.copy2(src, dst)
else: ACTIVE_DATA_DIR = DATA_DIR
sys.path.insert(0, str(PROJECT_ROOT))
from src.upstream_models import build_upstream_model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_upstream_model('transnet', PROJECT_ROOT).to(device)
with torch.no_grad(): output = model(torch.rand(2, 2, 32, 32, device=device))
assert output.shape == (2, 2, 32, 32), output.shape
print('TransNet forward OK | device:', device, '| parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
RUN_FULL_TRAINING = False  # 100 epoch 학습을 직접 시작할 때만 True로 바꾸세요.
EPOCHS = 100
if RUN_FULL_TRAINING:
    from src.train_baseline import train_model
    checkpoint = train_model('transnet', PROJECT_ROOT, ACTIVE_DATA_DIR, epochs=EPOCHS, batch_size=200, seed=42)
else:
    print('학습하지 않았습니다. 설정과 GPU를 확인한 후 직접 시작하세요.')

## 결과 위치

학습 시 `runs/transnet_indoor_cr4_100ep/`에 validation 기준 `best.pt`, epoch별 `history.csv`, 요약 `summary.json`이 생성됩니다. TransNet 원 저장소는 constant LR `1e-4`로 400/1000 epoch 결과를 보고하므로, 여기의 100 epoch는 빠른 경향성 확인용이지 공식 재현이 아닙니다.